In [ ]:
import pandas as pd
import os

from PersonAI.anchor import data_dir

# data path
DATA_PATH = data_dir()


# Load the CSV files

cozie_df = pd.read_csv(os.path.join(DATA_PATH,"cozie_oura_output_cleaned.csv"))
aircontrol_df = pd.read_csv(os.path.join(DATA_PATH,"Final_Aircontrol_Data.csv"))

# Get unique IDs
cozie_ids = set(cozie_df["id_participant"].unique())
aircontrol_ids = set(aircontrol_df["user_id"].unique())

# Compare
only_in_cozie = cozie_ids - aircontrol_ids
only_in_aircontrol = aircontrol_ids - cozie_ids
in_both = cozie_ids & aircontrol_ids

# Print results
print("Unique id_participant in cozie_oura_output:", len(cozie_ids))
print("Unique user_id in Final_Aircontrol_Data:", len(aircontrol_ids))
print("\nIDs only in cozie_oura_output:", only_in_cozie)
print("\nIDs only in Final_Aircontrol_Data:", only_in_aircontrol)
print("\nIDs in both:", in_both)



Unique id_participant in cozie_oura_output: 73
Unique user_id in Final_Aircontrol_Data: 73

IDs only in cozie_oura_output: set()

IDs only in Final_Aircontrol_Data: set()

IDs in both: {'id008', 'id082', 'id081', 'id061', 'id052', 'id074', 'id075', 'id042', 'id020', 'id016', 'id021', 'id006', 'id039', 'id046', 'id001', 'id048', 'id080', 'id036', 'id030', 'id055', 'id027', 'id033', 'id044', 'id053', 'id050', 'id005', 'id011', 'id010', 'id026', 'id028', 'id043', 'id056', 'id063', 'id065', 'id072', 'id002', 'id034', 'id076', 'id060', 'id045', 'id038', 'id085', 'id012', 'id009', 'id067', 'id058', 'id084', 'id022', 'id023', 'id032', 'id054', 'id007', 'id073', 'id051', 'id003', 'id071', 'id062', 'id059', 'id083', 'id078', 'id064', 'id049', 'id077', 'id037', 'id079', 'id013', 'id031', 'id017', 'id047', 'id057', 'id069', 'id024', 'id070'}


In [5]:
import re
from pathlib import Path
import pandas as pd
from pandas.api.types import DatetimeTZDtype

# --------- USER INPUTS ---------
BASE_DIR = Path("Data")

participants = {
    'id008','id082','id081','id061','id052','id074','id075','id042','id020','id016','id021','id006','id039','id046','id001','id048','id080','id036',
    'id030','id055','id027','id033','id044','id053','id050','id005','id011','id010','id026','id028','id043','id056','id063','id065','id072','id002',
    'id034','id076','id060','id045','id038','id085','id012','id009','id067','id058','id084','id022','id023','id032','id054','id007','id073','id051',
    'id003','id071','id062','id059','id083','id078','id064','id049','id077','id037','id079','id013','id031','id017','id047','id057','id069','id024','id070'
}

# Surveys must have all these answered
REQUIRED_Q_COLS = ["q_activity","q_feeling","q_heartbeat","q_location","q_preference_temperature"]

# ---- Merge groups (Cozie) ----
MERGE_GROUPS = {
    "heart_rate": ["ts_heart_rate", "ws_heart_rate"],
    "oxygen_saturation": ["ts_oxygen_saturation", "ws_oxygen_saturation"],
    "resting_heart_rate": ["ts_resting_heart_rate"],  # no ws counterpart given
    "hrv": ["ts_HRV", "ws_HRV"],
}

# Windows
WINDOWS_MIN = [3, 5, 10]   # general Cozie + Oura HR
WINDOWS_HOURS = [1, 3, 6]  # HRV-only

# --------- HELPERS ---------
def parse_cozie_all(cozie_path: Path) -> pd.DataFrame:
    """
    Load the FULL Cozie sample.csv.
    - Keep ALL rows (do NOT filter to surveys here).
    - Parse 'index' to timezone-aware UTC as column 'time'.
    - Keep original columns (including q_* and ts_*/ws_*).
    """
    df = pd.read_csv(cozie_path, low_memory=False)
    if "index" not in df.columns:
        raise ValueError(f"'index' column not found in {cozie_path}")
    df["time"] = pd.to_datetime(df["index"], utc=True, errors="coerce")
    # keep 'index' if you want to inspect original strings; it's harmless
    df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)
    return df

def extract_surveys(cozie_all: pd.DataFrame) -> pd.DataFrame:
    """
    Return ONLY the rows that are valid surveys (all q_* present), with a renamed
    survey time column 'RHRN_time' (copied from 'time') for clarity in outputs.
    """
    need_cols = [c for c in REQUIRED_Q_COLS if c in cozie_all.columns]
    if len(need_cols) != len(REQUIRED_Q_COLS):
        missing = set(REQUIRED_Q_COLS) - set(need_cols)
        raise ValueError(f"Missing required survey columns in Cozie file: {missing}")
    surveys = cozie_all.dropna(subset=need_cols).copy()
    surveys["RHRN_time"] = surveys["time"]
    surveys = surveys.sort_values("RHRN_time").reset_index(drop=True)
    return surveys

def find_oura_heartrate_csv(oura_dir: Path) -> Path | None:
    if not oura_dir.exists():
        return None
    for p in sorted(oura_dir.glob("*.csv")):
        if re.match(r"(?i)^heartrate.*\.csv$", p.name):
            return p
    return None

def parse_oura_heartrate(hr_path: Path) -> pd.DataFrame | None:
    if hr_path is None or not hr_path.exists():
        return None
    df = pd.read_csv(hr_path, low_memory=False)
    df.columns = [c.strip().lower() for c in df.columns]
    if "timestamp" not in df.columns:
        raise ValueError(f"'timestamp' column not found in {hr_path}")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    if "bpm" not in df.columns:
        df["bpm"] = pd.NA
    return df

def strip_tz_if_needed(df: pd.DataFrame, col: str) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out[col].dtype, DatetimeTZDtype):
        out[col] = out[col].dt.tz_localize(None)
    return out

def get_excel_writer(path):
    try:
        return pd.ExcelWriter(path, engine="xlsxwriter")
    except ModuleNotFoundError:
        print("[WARN] xlsxwriter not installed. Falling back to openpyxl (no colors). Run `pip install xlsxwriter` for colored tables.")
        return pd.ExcelWriter(path, engine="openpyxl")

def apply_boolean_colors(writer, sheet_name, df, suffix):
    if writer.engine != "xlsxwriter":
        return
    workbook  = writer.book
    worksheet = writer.sheets[sheet_name]
    green_fmt = workbook.add_format({"bg_color": "#C6EFCE", "font_color": "#006100"})
    red_fmt   = workbook.add_format({"bg_color": "#FFC7CE", "font_color": "#9C0006"})
    bool_cols = [c for c in df.columns if c.endswith(suffix)]
    if not bool_cols:
        return
    start_col = df.columns.get_loc(bool_cols[0])
    end_col   = df.columns.get_loc(bool_cols[-1])
    nrows = len(df)
    worksheet.conditional_format(1, start_col, nrows, end_col,
                                 {"type": "cell", "criteria": "==", "value": True, "format": green_fmt})
    worksheet.conditional_format(1, start_col, nrows, end_col,
                                 {"type": "cell", "criteria": "==", "value": False, "format": red_fmt})

# ---- Core: last reading before survey on MERGED streams (look up in full timeseries) ----
def last_gap_minutes_merged(ts_df: pd.DataFrame, time_col: str, value_cols: list[str], t_end: pd.Timestamp) -> float | None:
    present_cols = [c for c in value_cols if c in ts_df.columns]
    if not present_cols:
        return None
    mask_time = ts_df[time_col] <= t_end
    # any non-null among the merged columns counts as a reading
    mask_val  = ts_df[present_cols].notna().any(axis=1)
    mask = mask_time & mask_val
    if not mask.any():
        return None
    last_time = ts_df.loc[mask, time_col].max()
    return max((t_end - last_time).total_seconds() / 60.0, 0.0)

def oura_last_gap_minutes(oura_df: pd.DataFrame | None, t_end: pd.Timestamp) -> float | None:
    if oura_df is None or oura_df.empty:
        return None
    mask = (oura_df["timestamp"] <= t_end) & (oura_df["bpm"].notna())
    if not mask.any():
        return None
    last_time = oura_df.loc[mask, "timestamp"].max()
    return max((t_end - last_time).total_seconds() / 60.0, 0.0)

# --------- MAIN ---------
for pid in sorted(participants):
    cozie_path = BASE_DIR / pid / "Cozie" / "sample.csv"
    if not cozie_path.exists():
        print(f"[WARN] Missing Cozie file for {pid}: {cozie_path}")
        continue

    try:
        cozie_all = parse_cozie_all(cozie_path)   # <-- FULL FILE, all rows
        cozie_surveys = extract_surveys(cozie_all)  # <-- ONLY survey anchor rows
    except Exception as e:
        print(f"[ERROR] Cozie parse/extract failed for {pid}: {e}")
        continue

    if cozie_surveys.empty:
        print(f"[INFO] No valid survey rows for {pid}.")
        continue

    # Full timeseries (Cozie) used for lookups
    cozie_all_ts = cozie_all  # has 'time' in UTC

    # Oura HR
    oura_dir = BASE_DIR / pid / "Oura"
    hr_csv = find_oura_heartrate_csv(oura_dir)
    oura_hr_df = parse_oura_heartrate(hr_csv) if hr_csv else None

    # Availability rows per survey
    rows_general = []  # for 3/5/10 minutes
    rows_hrv     = []  # for 1/3/6 hours

    for _, r in cozie_surveys.iterrows():
        t = r["RHRN_time"]  # tz-aware UTC

        # --- merged Cozie gaps (in minutes) on the FULL timeseries ---
        gap_hr   = last_gap_minutes_merged(cozie_all_ts, "time", MERGE_GROUPS["heart_rate"], t)
        gap_spo2 = last_gap_minutes_merged(cozie_all_ts, "time", MERGE_GROUPS["oxygen_saturation"], t)
        gap_rhr  = last_gap_minutes_merged(cozie_all_ts, "time", MERGE_GROUPS["resting_heart_rate"], t)
        gap_hrv  = last_gap_minutes_merged(cozie_all_ts, "time", MERGE_GROUPS["hrv"], t)

        # --- Oura HR gap (in minutes) ---
        gap_oura = oura_last_gap_minutes(oura_hr_df, t)

        # ---------- general minutes windows ----------
        general_row = {"participant": pid, "RHRN_time": t}
        for minutes in WINDOWS_MIN:
            general_row[f"cozie_heart_rate_{minutes}min"]         = (gap_hr   is not None) and (gap_hr   <= minutes)
            general_row[f"cozie_oxygen_saturation_{minutes}min"]  = (gap_spo2 is not None) and (gap_spo2 <= minutes)
            general_row[f"cozie_resting_heart_rate_{minutes}min"] = (gap_rhr  is not None) and (gap_rhr  <= minutes)
            general_row[f"oura_heartrate_{minutes}min"]           = (gap_oura is not None) and (gap_oura <= minutes)
        rows_general.append(general_row)

        # ---------- HRV hour windows ----------
        hrv_row = {"participant": pid, "RHRN_time": t}
        for hours in WINDOWS_HOURS:
            minutes = hours * 60
            hrv_row[f"cozie_hrv_{hours}h"] = (gap_hrv is not None) and (gap_hrv <= minutes)
        rows_hrv.append(hrv_row)

    # ---- Write Excel (same file with multiple sheets) ----
    out_xlsx = BASE_DIR / pid / f"{pid}_availability.xlsx"
    with get_excel_writer(out_xlsx) as writer:
        # General sheets
        df_general = pd.DataFrame(rows_general).sort_values("RHRN_time")
        df_general_write = strip_tz_if_needed(df_general, "RHRN_time")
        for minutes in WINDOWS_MIN:
            sheet = f"{minutes}min"
            cols = ["participant","RHRN_time",
                    f"cozie_heart_rate_{minutes}min",
                    f"cozie_oxygen_saturation_{minutes}min",
                    f"cozie_resting_heart_rate_{minutes}min",
                    f"oura_heartrate_{minutes}min"]
            sub = df_general_write[cols]
            sub.to_excel(writer, sheet_name=sheet, index=False)
            apply_boolean_colors(writer, sheet, sub, suffix=f"_{minutes}min")

        # HRV sheets
        df_hrv = pd.DataFrame(rows_hrv).sort_values("RHRN_time")
        df_hrv_write = strip_tz_if_needed(df_hrv, "RHRN_time")
        for hours in WINDOWS_HOURS:
            sheet = f"HRV_{hours}h"
            cols = ["participant","RHRN_time", f"cozie_hrv_{hours}h"]
            sub = df_hrv_write[cols]
            sub.to_excel(writer, sheet_name=sheet, index=False)
            apply_boolean_colors(writer, sheet, sub, suffix=f"_{hours}h")

    print(f"[OK] Availability tables saved for {pid}: {out_xlsx}")


[OK] Availability tables saved for id001: Data\id001\id001_availability.xlsx
[OK] Availability tables saved for id002: Data\id002\id002_availability.xlsx
[OK] Availability tables saved for id003: Data\id003\id003_availability.xlsx
[OK] Availability tables saved for id005: Data\id005\id005_availability.xlsx
[OK] Availability tables saved for id006: Data\id006\id006_availability.xlsx
[OK] Availability tables saved for id007: Data\id007\id007_availability.xlsx
[OK] Availability tables saved for id008: Data\id008\id008_availability.xlsx
[OK] Availability tables saved for id009: Data\id009\id009_availability.xlsx
[OK] Availability tables saved for id010: Data\id010\id010_availability.xlsx
[OK] Availability tables saved for id011: Data\id011\id011_availability.xlsx
[OK] Availability tables saved for id012: Data\id012\id012_availability.xlsx
[OK] Availability tables saved for id013: Data\id013\id013_availability.xlsx
[OK] Availability tables saved for id016: Data\id016\id016_availability.xlsx

In [2]:
import re
import shutil
import io
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- CONFIG ----------
AVAIL_DIR = Path("Data/Availability")
SHEET_ORDER = ["3min", "5min", "10min", "HRV_1h", "HRV_3h", "HRV_6h"]
EXCLUDE_COLS = {"participant", "RHRN_time"}  # not part of metrics

# ---------- HELPERS ----------
def coerce_to_bool(s: pd.Series) -> pd.Series:
    """Robustly convert a column to boolean for %True calculation."""
    if s.dtype == bool:
        return s
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0) != 0
    lower = s.astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}
    return lower.map(mapping).fillna(False)

def sheet_percentages(df: pd.DataFrame) -> pd.DataFrame:
    """Compute percent True per metric column in one sheet."""
    cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    rows = []
    for c in cols:
        ser = coerce_to_bool(df[c])
        total = len(ser)
        true_n = int(ser.sum()) if total else 0
        pct = (true_n / total * 100.0) if total else 0.0
        rows.append({"metric": c, "percent_true": pct, "n_true": true_n, "n_rows": total})
    out = pd.DataFrame(rows).sort_values("percent_true", ascending=False).reset_index(drop=True)
    return out

def sort_key(label: str):
    """Sort minutes before hours; then by amount; then by variable name."""
    import re as _re
    m = _re.search(r"_(\d+)(min|h)$", label)
    if not m:
        return (9999, 9999, label)
    amount = int(m.group(1))
    unit = m.group(2)
    unit_rank = 0 if unit == "min" else 1
    var = label[: label.rfind("_")]
    return (unit_rank, amount, var)

def fig_to_bytesio(fig) -> io.BytesIO:
    """Render a matplotlib figure to a PNG in memory and return as BytesIO."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf

def make_barh_image(df: pd.DataFrame, title: str) -> io.BytesIO | None:
    """Create a horizontal bar chart image (as BytesIO)."""
    if df is None or df.empty:
        return None
    labels = df["metric"].tolist()
    values = df["percent_true"].astype(float).tolist()

    H = max(4, 0.35 * len(labels))
    fig = plt.figure(figsize=(12, H))
    ax = fig.add_subplot(111)
    y = np.arange(len(labels))
    bars = ax.barh(y, values)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Percent True")
    ax.set_title(title)
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.set_xlim(0, 100)
    # add labels
    for bar, val in zip(bars, values):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.1f}%",
                va="center", ha="left", fontsize=9)
    fig.tight_layout()
    return fig_to_bytesio(fig)

# ---------- MAIN ----------
workbooks = sorted(AVAIL_DIR.glob("*_availability.xlsx"))
if not workbooks:
    print(f"[WARN] No availability files found in {AVAIL_DIR.resolve()}")

for xlsx in workbooks:
    # derive participant id
    m = re.search(r"(id\d+)_availability\.xlsx$", xlsx.name, flags=re.IGNORECASE)
    pid = (m.group(1).lower() if m else xlsx.stem)

    # ensure participant folder exists
    pdir = AVAIL_DIR / pid
    pdir.mkdir(parents=True, exist_ok=True)

    # ---- READ original workbook safely (and close immediately after) ----
    per_sheet_tables: dict[str, pd.DataFrame] = {}
    with pd.ExcelFile(xlsx) as xls:
        sheet_names = xls.sheet_names
        for sheet in SHEET_ORDER:
            if sheet in sheet_names:
                df = xls.parse(sheet_name=sheet)
                per_sheet_tables[sheet] = sheet_percentages(df)
            else:
                per_sheet_tables[sheet] = pd.DataFrame(columns=["metric", "percent_true", "n_true", "n_rows"])

    # ---- WRITE percentages workbook safely (and embed charts from memory) ----
    pct_xlsx_path = pdir / f"{pid}_availability_percentages.xlsx"
    with pd.ExcelWriter(pct_xlsx_path, engine="xlsxwriter") as writer:
        workbook = writer.book
        for sheet in SHEET_ORDER:
            stats_df = per_sheet_tables.get(sheet, pd.DataFrame(columns=["metric", "percent_true", "n_true", "n_rows"]))
            # write table
            stats_df.to_excel(writer, sheet_name=sheet, index=False)
            ws = writer.sheets[sheet]
            ws.set_column(0, 0, 42)    # metric column width
            ws.set_column(1, 3, 14)    # percent & counts

            # embed chart image (no temp files)
            img_buf = make_barh_image(stats_df, f"{pid} — {sheet}")
            if img_buf is not None:
                # place chart to the right of the table
                ws.insert_image("F2", "chart.png", {"image_data": img_buf})

    # ---- SAVE one combined PNG (all metrics across sheets), no file handles kept open ----
    all_stats = pd.concat(per_sheet_tables.values(), ignore_index=True)
    # stable metric ordering across windows
    all_stats["sortkey"] = all_stats["metric"].map(sort_key)
    all_stats = all_stats.sort_values(["sortkey", "percent_true"], ascending=[True, False]).drop(columns=["sortkey"])
    combined_img = make_barh_image(all_stats, f"{pid} — Availability % True (all metrics)")
    combined_png_path = pdir / f"{pid}_availability_all.png"
    if combined_img is not None:
        with open(combined_png_path, "wb") as f:
            f.write(combined_img.getbuffer())

    # ---- MOVE original workbook AFTER all reads/writes are closed ----
    dest_xlsx = pdir / xlsx.name
    if xlsx.resolve() != dest_xlsx.resolve():
        try:
            shutil.move(str(xlsx), str(dest_xlsx))
        except PermissionError:
            # fallback: copy then try to delete (Excel may still have it open)
            shutil.copy2(str(xlsx), str(dest_xlsx))
            try:
                xlsx.unlink()
            except PermissionError:
                print(f"[WARN] Could not delete source (locked by another app): {xlsx}")

    print(f"[OK] {pid}:")
    print(f"     - Original moved -> {dest_xlsx.relative_to(AVAIL_DIR)}")
    print(f"     - Percentages    -> {pct_xlsx_path.relative_to(AVAIL_DIR)}")
    if combined_img is not None:
        print(f"     - Combined chart -> {combined_png_path.relative_to(AVAIL_DIR)}")

print("[DONE] All participants processed without leaving files open.")


[OK] id002:
     - Original moved -> id002\id002_availability.xlsx
     - Percentages    -> id002\id002_availability_percentages.xlsx
     - Combined chart -> id002\id002_availability_all.png
[OK] id003:
     - Original moved -> id003\id003_availability.xlsx
     - Percentages    -> id003\id003_availability_percentages.xlsx
     - Combined chart -> id003\id003_availability_all.png
[OK] id005:
     - Original moved -> id005\id005_availability.xlsx
     - Percentages    -> id005\id005_availability_percentages.xlsx
     - Combined chart -> id005\id005_availability_all.png
[OK] id006:
     - Original moved -> id006\id006_availability.xlsx
     - Percentages    -> id006\id006_availability_percentages.xlsx
     - Combined chart -> id006\id006_availability_all.png
[OK] id007:
     - Original moved -> id007\id007_availability.xlsx
     - Percentages    -> id007\id007_availability_percentages.xlsx
     - Combined chart -> id007\id007_availability_all.png
[OK] id008:
     - Original moved -> id0